<a href="https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes



In [2]:
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get("PurpleElegantBass749671")
print(f"Token loaded: {hf_token[:6]}... (length {len(hf_token)})" if hf_token else "Token is empty/None!")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
DECISION_MOMENT = "2026-04-30"
WINDOW_START = "2026-01-30"
SPLIT_DATE = "2026-03-15"

#read the daily content performance data from Parquet files covering January through April 2026
#filter it to the specific feature window between WINDOW_START and DECISION_MOMENT
#group the data by client and page
#split impressions into an early period (imp_early) and a later period (imp_late) based on SPLIT_DATE
#calculates total impressions, total clicks, and average search position for the entire window
#produce a DataFrame with one row per client-page combination

feature_paths = [f"{rel}/fact_content_daily_performance/month=2026-0{m}/*.parquet" for m in [1, 2, 3, 4]]

feature_df = con.sql(f"""
    WITH daily AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet([{', '.join(f"'{p}'" for p in feature_paths)}])
        WHERE report_date BETWEEN DATE '{WINDOW_START}' AND DATE '{DECISION_MOMENT}'
    )


    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '{SPLIT_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_early,
        SUM(CASE WHEN report_date >  DATE '{SPLIT_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_late,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,
        AVG(gsc_avg_position) AS avg_position_90d
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

qualifying = con.sql(f"""
    SELECT c.content_hash_id, c.client_hash_id, c.content_type, c.main_intent,
           c.word_count, c.char_count, c.competition_level, c.search_volume
    FROM read_parquet('{rel}/dim_content.parquet') c
    JOIN read_parquet('{rel}/dim_clients.parquet') cl USING (client_hash_id)
    WHERE c.content_created_date <= DATE '{WINDOW_START}'
      AND cl.gsc_data_start <= DATE '{WINDOW_START}'
""").df()

feature_df = feature_df.merge(qualifying, on=["client_hash_id", "content_hash_id"], how="inner")
feature_df["ctr"] = np.where(
    feature_df["impressions_90d"] > 0,
    (feature_df["clicks_90d"] / feature_df["impressions_90d"]) * 100,
    np.nan,
)
feature_df["is_declining"] = (feature_df["imp_late"] < feature_df["imp_early"]).astype(int)

print(f"Rows: {len(feature_df):,}")
feature_df.head()

Token loaded: hf_IcB... (length 37)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 260,285


,client_hash_id,content_hash_id,imp_early,imp_late,impressions_90d,clicks_90d,avg_position_90d,content_type,main_intent,word_count,char_count,competition_level,search_volume,ctr,is_declining
0,client_3ffa76342f366962,content_5067dafdab76f97c,0.0,0.0,0.0,0.0,NaN,feedly article,None,850,6221,None,<NA>,NaN,0
1,client_3ffa76342f366962,content_cfd5f8764543bfee,1.0,0.0,1.0,0.0,5.0,feedly article,None,640,4551,None,<NA>,0.0,1
2,client_3ffa76342f366962,content_6d26e2f1a8a6512b,1.0,0.0,1.0,0.0,36.0,feedly article,None,706,4905,None,<NA>,0.0,1
3,client_3ffa76342f366962,content_ab514679d7b70992,0.0,0.0,0.0,0.0,NaN,feedly article,None,736,5049,None,<NA>,NaN,0
4,client_3ffa76342f366962,content_f1a36b8d0b297a35,0.0,0.0,0.0,0.0,NaN,feedly article,None,694,5165,None,<NA>,NaN,0


1. **Declining with demand** (`imp_early` vs `imp_late`) — the warehouse-honest replacement for the CSV's staleness signal. The idea: `dim_content.content_updated_date` looked like a natural staleness signal but turned out to be a snapshot field that systematically mislabels actively-maintained pages (see `w03_feature_leakage_check.ipynb`, leakage hunt). A page whose impressions genuinely dropped within the observed 90-day window, split in half, is a real, non-leaky decline signal instead.
2. **CTR-vs-position** (`ctr` by position bucket) — same signal as the CSV, rebuilt from `gsc_clicks`/`gsc_impressions`/`gsc_avg_position`.

For signal 1, the check-only label was **May 2026's impression rate vs the late-window rate** — never a rule input, only used to test whether the signal is real. This mirrors the CSV's approach of checking staleness against `is_declining_label` without ever using the label in the rule.

**Signal 1 check — declining with demand vs May outcome** (`impressions_90d >= 200` demand floor, rates compared to correct for the 46-day vs 31-day window mismatch):

| Group | Still declining into May | n |
|---|---|---|
| Declining in-window | 60.7% | 42,074 |
| Not declining, but still visible | 57.6% | 38,122 |

**Verdict: MIXED.** The 3-point gap is real given n≈40k in each group (many standard errors from chance), but it's small — a "declining" page is only modestly more likely to keep declining than a comparable page that wasn't flagged. Directional, not strongly predictive on its own — the same MIXED call the CSV's staleness signal got, for a similar reason (a real but thin effect).

**Signal 2 check — CTR by position bucket** (rows with `impressions_90d = 0` excluded — no position/CTR data to bucket):

| Bucket | Mean CTR | n |
|---|---|---|
| top_3 | 1.64% | 8,451 |
| page_1 | 0.61% | 63,186 |
| striking | 0.34% | 32,292 |
| page_3_5 | 0.22% | 31,988 |
| deep | 0.13% | 10,339 |


**Verdict: CONFIRMED.** Cleanly monotonic, thousands of rows per bucket — the strongest signal available, same as the CSV finding.

**The rule, in plain words**

> A page is worth a review if it has real, current search demand, it ranks somewhere a reviewer could expect a reasonable CTR, its actual CTR falls clearly short of that, *and* its own impressions dropped within the observed window rather than holding steady or growing.

- **Score** (readable on purpose): `score = visible * position_ok * ctr_low * declining * impressions_90d`
- **Reason code:** `declining_ctr_gap`
- **Action label:** `refresh_review`

**Thresholds, tightened from the first pass:** `impressions_90d >= 500` (visibility floor, matches the CSV's own low-CTR flag convention), `0 < avg_position_90d <= 20` (position where a better CTR is realistic), `ctr < 0.4` (tightened from 0.5 — sits closer to the `striking` tier's 0.34% mean than the `page_1` tier's 0.61%, so fewer borderline page-1 pages get swept in as false positives), `is_declining == 1` (impressions dropped from the early half of the window to the late half).

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import os
import numpy as np

visible = feature_df["impressions_90d"] >= 500
position_ok = (feature_df["avg_position_90d"] > 0) & (feature_df["avg_position_90d"] <= 20)
ctr_low = feature_df["ctr"] < 0.4
declining = feature_df["is_declining"] == 1

flag = visible & position_ok & ctr_low & declining

feature_df["score"] = np.where(flag, feature_df["impressions_90d"], 0)
feature_df["reason_code"] = np.where(flag, "declining_ctr_gap", "not_flagged")
feature_df["action"] = np.where(flag, "refresh_review", "monitor")

queue_cols = [
    "content_hash_id", "client_hash_id", "score", "reason_code", "action",
    "impressions_90d", "avg_position_90d", "ctr", "imp_early", "imp_late",
    "content_type", "main_intent",
]
ranked = feature_df.sort_values("score", ascending=False).reset_index(drop=True)[queue_cols]

os.makedirs("../outputs", exist_ok=True)
ranked.to_csv("../outputs/baseline_action_score_warehouse.csv", index=False)

print(f"Rows written: {len(ranked):,}")
print(f"Rows flagged (score > 0): {(ranked['score'] > 0).sum():,}")
ranked.head(20)

Rows written: 260,285
Rows flagged (score > 0): 22,971


,content_hash_id,client_hash_id,score,reason_code,action,impressions_90d,avg_position_90d,ctr,imp_early,imp_late,content_type,main_intent
0,content_e8a52cf3d5988c07,client_23a62021009f63c4,587268.0,declining_ctr_gap,refresh_review,587268.0,13.487026,0.327959,311811.0,275457.0,keyword article,transactional
1,content_b99ea6861864dea5,client_62f4a7e64f5e0096,504886.0,declining_ctr_gap,refresh_review,504886.0,5.029983,0.162215,260663.0,244223.0,keyword article,informational
2,content_8e1334d6356668e3,client_73cda7b4e4f265ea,467252.0,declining_ctr_gap,refresh_review,467252.0,5.375914,0.000642,262729.0,204523.0,keyword article,commercial
3,content_fec55986a1868d62,client_73cda7b4e4f265ea,396139.0,declining_ctr_gap,refresh_review,396139.0,7.063606,0.000252,204152.0,191987.0,keyword article,informational
4,content_7c6373141eae744a,client_62f4a7e64f5e0096,302645.0,declining_ctr_gap,refresh_review,302645.0,5.632297,0.071701,207565.0,95080.0,keyword article,commercial
5,content_db122b8ba22641b8,client_73cda7b4e4f265ea,299393.0,declining_ctr_gap,refresh_review,299393.0,4.325950,0.280902,174132.0,125261.0,keyword article,commercial
6,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,279677.0,declining_ctr_gap,refresh_review,279677.0,9.509454,0.000715,279427.0,250.0,keyword article,informational
7,content_5e1c049f62e33b11,client_23a62021009f63c4,276662.0,declining_ctr_gap,refresh_review,276662.0,17.040995,0.146749,148979.0,127683.0,keyword article,transactional
8,content_62850cc56d45a7cf,client_62f4a7e64f5e0096,240905.0,declining_ctr_gap,refresh_review,240905.0,5.258431,0.288080,123171.0,117734.0,keyword article,informational
9,content_e0ca055423cbe896,client_62f4a7e64f5e0096,237099.0,declining_ctr_gap,refresh_review,237099.0,3.222487,0.284269,132319.0,104780.0,keyword article,informational


In [4]:
print(f"Total flagged (score > 0): {(ranked['score'] > 0).sum():,} / {len(ranked):,}")
print(ranked.head(20)["client_hash_id"].value_counts())

Total flagged (score > 0): 22,971 / 260,285
client_hash_id
client_62f4a7e64f5e0096    11
client_73cda7b4e4f265ea     6
client_23a62021009f63c4     2
client_e547b89c05043229     1
Name: count, dtype: int64



## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**I can't write this section for you** — it needs the actual top-20 `content_hash_id` rows, which only exist once you run the cell above in Colab against the real warehouse (my sandbox can't reach Hugging Face). Once you've run it, look at `ranked.head(20)` and write 1–2 lines per row: what makes it a plausible pick, and what would make it wrong (same style as the CSV version — e.g. *"wrong if the query is heavily zero-click / a SERP feature is absorbing clicks rather than the page failing"*). I'm happy to draft this with you once you paste back the real top-20 output.

**Top-20 review**

All 20 rows are `content_type = keyword article` — this baseline is entirely reviewing
programmatic keyword content, not editorial pages. Worth remembering when reading the rest:
we don't yet know if the rule generalizes beyond this content type.

**The typical case (16 of 20 rows).** High real demand (impressions_90d 175K–590K), page-1
or near-page-1 position (avg_position 3–17), and a genuine decline within the window
(early-to-late drops of roughly 10–40%). CTR sits well below what Section 1's bucket check
says a page at that position should earn — e.g. row 0 (position 13.5, in the "striking"
zone) has a 0.33% CTR, close to that bucket's 0.34% mean, but its neighbors at similar or
better positions (rows 8–19) run 0.10–0.29%, below what their position tier would predict.
This is the profile the rule is designed to catch: real traffic, real ranking, clicks falling
short. Wrong if: the CTR gap is a SERP-feature effect (a featured snippet or People Also Ask
box absorbing clicks) rather than the page itself underperforming — avg_position measures a
slot in results, not whether a searcher ever saw a clickable blue link.

**Outlier 1 — row 6 (`content_9c057b66c30a3abb`).** Impressions collapsed from 279,427 to 250
within the window — a >99.9% drop, not a gradual decline, with CTR effectively zero (0.0007%).
This doesn't look like a content-quality problem; it looks like the page may have dropped out
of the index, hit a canonical/redirect issue, or lost tracking entirely. Wrong if: exactly
that — a technical/indexing issue a content rewrite won't fix. This one probably needs a
technical check before a content review.

**Outlier 2 — rows 2 and 3 (`client_73cda7b4e4f265ea`).** Both rank strongly (avg_position
5.4 and 7.1 — solid page 1) but CTR is near zero (0.06%, 0.03%) — well below even the "deep"
position tier's 0.13% average from Section 1, despite ranking far better than that tier.
That gap is too large to be normal underperformance. Wrong if: a competing result or SERP
feature is intercepting clicks before they'd reach this page, which a content refresh
wouldn't fix either.

**Client concentration.** 17 of 20 rows come from 3 clients (`client_62f4a7e64f5e0096`,
`client_73cda7b4e4f265ea`, `client_23a62021009f63c4`) — see Section 4 for the full pattern
and why it matters for a real reviewer handoff.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# Weak-pick pattern 1: client concentration in the top 20
print("Top-20 client concentration:")
print(ranked.head(20)["client_hash_id"].value_counts())

# Weak-pick pattern 2: how much of the 90-day window did the flagged rows actually have data for?
# (a page that barely existed in the window could look 'declining' from very little signal)
flagged_ids = ranked[ranked["score"] > 0]["content_hash_id"]
print(f"\nFlagged rows: {len(flagged_ids):,}")

# Leakage check: confirm no excluded/future-window columns feed the rule
banned = ["content_updated_date", "last_optimized_date", "optimization_eligible_date",
          "is_published", "is_deleted", "impressions_may", "rate_may", "still_declining_may"]
rule_inputs = ["impressions_90d", "avg_position_90d", "ctr", "imp_early", "imp_late"]
leaked = [c for c in rule_inputs if c in banned]
print(f"\nRule inputs: {rule_inputs}")
print(f"Any leaked/future-window/product-flag inputs used in the rule? {bool(leaked)}")

Top-20 client concentration:
client_hash_id
client_62f4a7e64f5e0096    11
client_73cda7b4e4f265ea     6
client_23a62021009f63c4     2
client_e547b89c05043229     1
Name: count, dtype: int64

Flagged rows: 22,971

Rule inputs: ['impressions_90d', 'avg_position_90d', 'ctr', 'imp_early', 'imp_late']
Any leaked/future-window/product-flag inputs used in the rule? False


**What the checks turned up**

22,971 of 260,285 rows (8.8%) were flagged — a reasonable middle ground, not too strict or too loose.

Client concentration: the top 20 flagged rows come from only 4 of the qualifying clients.
`client_62f4a7e64f5e0096` alone takes 11 of the top 20 (55%), `client_73cda7b4e4f265ea` takes
6 more — together 85% of the top 20 from two clients. This is the same pattern the Week 4 CSV
baseline showed (one client took 8 of its top 10 picks) — a real, recurring limitation rather
than a one-off. A reviewer handoff built on this score needs a per-client cap on the queue,
or the queue will keep surfacing the same one or two large accounts regardless of whether the
score comes from a rule or (later) a model.

**Leakage check.** The rule's five inputs (`impressions_90d`, `avg_position_90d`, `ctr`,
`imp_early`, `imp_late`) are all observed within the 2026-01-30–2026-04-30 feature window,
all knowable at the decision moment. None of `content_updated_date`, `last_optimized_date`,
`optimization_eligible_date`, the product state flags, or any May/June data went into the
score, reason code, or action label — May was used only as a check-only label in
`w03_feature_leakage_check.ipynb`, never here.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.